# End of week 1 exercise

Build a Spanish tutor that creates import-ready flashcards for Flashcards.com.  
Each card has Spanish on the front and English on the back.

In [8]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [9]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [10]:
# set up environment

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
has_openai_key = bool(openai_api_key and openai_api_key.startswith("sk-"))

openai = OpenAI(api_key=openai_api_key) if has_openai_key else None
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

if has_openai_key:
    print("OpenAI API key found; GPT answer is enabled.")
else:
    print("No OpenAI API key found; GPT answer cell will explain how to enable it.")
print("For the Llama answer, make sure `ollama serve` is running.")

No OpenAI API key found; GPT answer cell will explain how to enable it.
For the Llama answer, make sure `ollama serve` is running.


In [ ]:
# Describe the flashcards you want; type over this for each new deck

question = """
"""

In [12]:
# Get gpt-4o-mini to create the flashcards, with streaming

system_prompt = """
You are a precise Spanish tutor and flashcard generator.
Create a CSV deck for importing into Flashcards.com.
The first column is the Spanish front; the second column is the English back.
Return no explanation, no Markdown, no code fences, and no text before or after the CSV.
Return exactly one header row: Spanish,English, followed by no more than 50 card rows.
Every card must contain exactly one Spanish-English meaning pair with article.
Quote every field with double quotes and escape any internal double quote by doubling it.
Use natural, useful language and correct accents. Do not number the cards.
"""

user_prompt = f"""
Create the flashcard CSV for this request. Use at most 50 cards:

{question}
""" 

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

if not has_openai_key:
    display(Markdown(
        "OpenAI is skipped because `OPENAI_API_KEY` is blank in `.env`. "
        "Add a valid key if you want the GPT answer. The next cell still runs locally with Ollama."
    ))
else:
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True,
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

OpenAI is skipped because `OPENAI_API_KEY` is blank in `.env`. Add a valid key if you want the GPT answer. The next cell still runs locally with Ollama.

In [13]:
# Get Llama 3.2 to create the same flashcard CSV locally

try:
    response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
    reply = response.choices[0].message.content
    display(Markdown("```csv\n" + reply.strip() + "\n```"))
except Exception as error:
    display(Markdown(
        "**Ollama is not reachable.** Start it in a terminal with `ollama serve`,"
        " then run this cell again.\n\n"
        f"Technical detail: `{type(error).__name__}`"
    ))

```csv
"Spanish,English",
"Lechuga", "Lettuce"
"Ajo", "Garlic"
"Pepino", "Cucumber"
"Zanahoria", "Carrot"
"Remolacha", "Cabbage"
"Pimiento", "Pepper"
"Chilindrón", "Onion"
" Aguacate", "Avocado"
" Espinaca", "Spinach"
" Col", "Kohlrabi"
" Chardón", "Cardoon"
" Berenjena", "Eggplant"
" Tomate", "Tomato"
"Salsa", "Sauc"
"Pimiento rojo", "Red Pepper"
"Pimiento green", "Green Pepper"
" Cebolla", "Chives"
" Maíz", "Corn"
" Champiñones", "Mushrooms"
" Aguacate maduro", "Ripe Avocado"
" Alcaparra", "Lemon"
" Pimiento encurtido", "Pickled Pepper"
" Espinaca congarada", "Spicy Spinach"
" Chardón arrugado", "Wrinkled Cardoon"
" Remolacha blanca", "White Cabbage"
" Aguacate agrietado", "Spoiled Avocado"
```